<p style="text-align:center"><a href="https://skills.network" target="_blank"><img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"></a></p>

# **Launch Sites Locations Analysis with Folium**

Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and the location of a launch site.
In this lab, you will perform interactive visual analytics using `Folium`.

## Objectives

- **TASK 1:** Mark all launch sites on a map
- **TASK 2:** Mark the success/failed launches for each site on the map
- **TASK 3:** Calculate the distances between a launch site to its proximities


In [1]:
!pip3 install folium wget pandas

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9692 sha256=be1dac36c79121525a8bf4814198f035fbe7325eba5548cb988b9b104ab50987
  Stored in directory: c:\users\dell\appdata\local\pip\cache\wheels\9c\b9\25\66d2377ed05ab1424fa6b361f1088fc5ae065f96efad7202dc
Successfully built wget

   ------ --------------------------------- 1/6 [xyzservices]
   -------------------- ------------------- 3/6 [jinja2]
   -------------------- ------------------- 3/6 [jinja2]
   -------------------- ------------------- 3/6 [jinja2]
   -------------------- ------------------- 3/6 [jinja2]
   -------------------- ------------------- 3/6 [jinja2]
   -------------------- ---


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import folium
import wget
import pandas as pd
from folium.plugins import MarkerCluster
from folium.plugins import MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

## Task 1: Mark all launch sites on a map


In [2]:
# Download and read the spacex_launch_geo.csv
spacex_csv_file = wget.download('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
spacex_df = pd.read_csv(spacex_csv_file)

# Select relevant columns
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [4]:
# Start location: NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

# Create a circle at NASA Johnson Space Center
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
marker = folium.map.Marker(
    nasa_coordinate,
    icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
                 html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC')
)
site_map.add_child(circle)
site_map.add_child(marker)
site_map

In [5]:
# Initialize the map centered over the US
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# For each launch site, add a Circle and Marker
for index, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    site_name = row['Launch Site']
    
    # Add circle
    folium.Circle(
        coordinate,
        radius=1000,
        color='#000000',
        fill=True
    ).add_child(folium.Popup(site_name)).add_to(site_map)
    
    # Add text marker
    folium.map.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % site_name
        )
    ).add_to(site_map)

site_map

# Task 2: Mark the success/failed launches for each site on the map


In [6]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


In [7]:
# Function to assign color to launch outcome
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'

spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


In [8]:
marker_cluster = MarkerCluster()

# Add marker_cluster to site_map
site_map.add_child(marker_cluster)

# For each launch record, add a Marker to marker_cluster
for index, row in spacex_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    color = row['marker_color']
    
    folium.Marker(
        location=coordinate,
        icon=folium.Icon(color='white', icon_color=color, icon='circle', prefix='fa'),
        popup=folium.Popup(
            f"Launch Site: {row['Launch Site']}<br>Outcome: {'Success' if row['class'] == 1 else 'Failure'}",
            max_width=200
        )
    ).add_to(marker_cluster)

site_map

# TASK 3: Calculate the distances between a launch site to its proximities


In [9]:
# Add MousePosition to get coordinates on hover
formatter = "function(num) {return L.Util.formatNum(num, 5);}"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter
)
site_map.add_child(mouse_position)
site_map

In [10]:
def calculate_distance(lat1, lon1, lat2, lon2):
    """Calculate distance in km between two coordinates using Haversine formula."""
    R = 6373.0  # approximate radius of earth in km

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    distance = R * c
    return distance

In [11]:
# Launch site: KSC LC-39A
launch_site_lat = 28.573255
launch_site_lon = -80.646895

# Closest coastline point (Cape Canaveral coast)
coastline_lat = 28.56367
coastline_lon = -80.57163

distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
print(f'Distance to coastline: {distance_coastline:.2f} km')

Distance to coastline: 7.43 km


In [12]:
# Add marker on the coastline with distance label
distance_marker = folium.Marker(
    [coastline_lat, coastline_lon],
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html=f'<div style="font-size: 12; color:#d35400;"><b>{distance_coastline:.2f} km</b></div>'
    )
)
site_map.add_child(distance_marker)

In [13]:
# Draw a PolyLine from the launch site to the coastline
coordinates = [[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]]
lines = folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)
site_map

In [14]:
# Distance to closest city (Cocoa Beach, FL)
city_lat = 28.3200
city_lon = -80.6076
distance_city = calculate_distance(launch_site_lat, launch_site_lon, city_lat, city_lon)
print(f'Distance to nearest city (Cocoa Beach): {distance_city:.2f} km')

city_marker = folium.Marker(
    [city_lat, city_lon],
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html=f'<div style="font-size:12;color:blue;"><b>City: {distance_city:.2f} km</b></div>'
    )
)
site_map.add_child(city_marker)

city_line = folium.PolyLine([[launch_site_lat, launch_site_lon], [city_lat, city_lon]], weight=1, color='blue')
site_map.add_child(city_line)

# Distance to closest highway (US-1 highway)
highway_lat = 28.5620
highway_lon = -80.8037
distance_highway = calculate_distance(launch_site_lat, launch_site_lon, highway_lat, highway_lon)
print(f'Distance to nearest highway (US-1): {distance_highway:.2f} km')

highway_marker = folium.Marker(
    [highway_lat, highway_lon],
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html=f'<div style="font-size:12;color:purple;"><b>Highway: {distance_highway:.2f} km</b></div>'
    )
)
site_map.add_child(highway_marker)

highway_line = folium.PolyLine([[launch_site_lat, launch_site_lon], [highway_lat, highway_lon]], weight=1, color='purple')
site_map.add_child(highway_line)

site_map

Distance to nearest city (Cocoa Beach): 28.43 km
Distance to nearest highway (US-1): 15.37 km


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium.
Next, you will need to build a dashboard using Plotly Dash on detailed launch records.


## Authors

[Yan Luo](https://www.linkedin.com/in/yan-luo-96288783/)

**Completed by:** [Hrestak33](https://github.com/Hrestak33)

Copyright © 2021 IBM Corporation. All rights reserved.
